Usei o modelo  **SentenceTransformer('BAAI/bge-m3') **Mais preciso para semântica, melhor para sentimentos e similaridade o Tamanho do embedding é de 1024,significa que esse numero tem as caracteristicas semnatincas que é o significado do texto, quanto maior esse numero entendo que melhor vai ser o resultado do texto.Na utilização desse embedding o modelo teve uma Acurácia: 0.96, isso significa que o modelo ficou muito bom, pois o embedding ele tem um bom treinameto.





In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pathlib
import textwrap
import numpy as np
import pandas as pd
import os
#import google.generativeai as genai
import warnings
warnings.filterwarnings('ignore')
import gradio as gr

In [3]:

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# 🔹 Carrega o modelo local (só precisa ser feito uma vez)
#modelo = SentenceTransformer('all-MiniLM-L6-v2')
#modelo = SentenceTransformer('all-mpnet-base-v2')
modelo = SentenceTransformer('BAAI/bge-m3')




def gerar_embeddings(texto):
    """
    Gera embeddings de um texto usando modelo local SentenceTransformer.
    Retorna uma lista de floats (vetor numérico).
    """
    embedding = modelo.encode(texto)
    return embedding


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (7 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Descrições dos sentimentos associadas às suas categorias
sentiment_descriptions = {
    'positivo': [
        'Comentários positivos que expressam satisfação',
        'Chegou antes do prazo e a qualidade é excelente',
        'Excelente acabamento e encaixe perfeito.',
        'Ótima resistência e durabilidade.',
        'Produto superou minhas expectativas.',
        'Muito eficiente e fácil de instalar',
        'Ótimo custo-benefício.',
        'Entrega rápida e embalagem segura.',
        'atendimento a cima do esperado',
        'empresa boa',
        'Peça excelente',
        'produto com otima qualidade'

    ],
    'negativo': [
        'Barulho excessivo e vibração.',
        'Preço alto para a qualidade entregue',
        'Quebrou na primeira semana de uso',
        'Chegou sem manual e sem instruções de instalação',
        'produto demorou a chegar',
        'produto com defeito',
        'Falta de atendimento',
        'atendimento pessimo',
        'empresa não responde mensagens',
        'A peça chegou danificada e não pôde ser utilizada',
        'O prazo de entrega foi maior do que o previsto',
        'apresentou vazamento e precisou ser substituído',
        'aquece acima do esperado durante o uso',
        'peça com problema',
        'peça não chegou no prazo'
        'peça quebrou com pouco tempo de uso',
    ]#,
    #'neutro': [
     #   'Comentários neutros que expressam neutralidade ou falta de emoção em relação ao atendimento médico'
    #]
}

# Gerando os embeddings para cada descrição
embeddings_sentimentos = []
categoria_mapping = []
descricao_mapping = []

for categoria, descricoes in sentiment_descriptions.items():
    for descricao in descricoes:
        embedding = gerar_embeddings(descricao)
        embeddings_sentimentos.append(embedding)
        categoria_mapping.append(categoria)
        descricao_mapping.append(descricao)

In [ ]:
categoria_mapping

['positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo']

In [5]:
def classificar(comentario_paciente, verbose=False):
    embedding_paciente = np.array(gerar_embeddings(comentario_paciente)).reshape(1, -1)

    # Calcula a similaridade de cosseno para cada embedding
    similaridades = [cosine_similarity(embedding_paciente, np.array(embedding).reshape(1, -1))[0][0]
                     for embedding in embeddings_sentimentos]

    # Converte a lista de similaridades em um dataframe do pandas para facilitar a análise
    similaridades_df = pd.DataFrame({
        'Similaridade': similaridades,
        'Categoria': categoria_mapping,
        'Descricao': descricao_mapping
    })

    # Encontrando a categoria e descrição com maior similaridade
    idx_max_similaridade = similaridades_df['Similaridade'].idxmax()
    categoria_mais_similar = similaridades_df['Categoria'][idx_max_similaridade]
    descricao_mais_similar = similaridades_df['Descricao'][idx_max_similaridade]

    if verbose:
        print('------------------')
        print('Tabela de similaridades:')
        print(similaridades_df)
        print('------------------')
        print(f'Índice da descrição mais similar: {idx_max_similaridade}')
        print(f'Categoria mais similar: {categoria_mais_similar}')
        print(f'Descrição mais similar: {descricao_mais_similar}')
        print('------------------')

    return categoria_mais_similar

In [6]:
categoria_mapping

['positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'positivo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo',
 'negativo']

In [7]:
def classificar(comentario_paciente, verbose=False):
    embedding_paciente = np.array(gerar_embeddings(comentario_paciente)).reshape(1, -1)

    # Calcula a similaridade de cosseno para cada embedding
    similaridades = [cosine_similarity(embedding_paciente, np.array(embedding).reshape(1, -1))[0][0]
                     for embedding in embeddings_sentimentos]

    # Converte a lista de similaridades em um dataframe do pandas para facilitar a análise
    similaridades_df = pd.DataFrame({
        'Similaridade': similaridades,
        'Categoria': categoria_mapping,
        'Descricao': descricao_mapping
    })

    # Encontrando a categoria e descrição com maior similaridade
    idx_max_similaridade = similaridades_df['Similaridade'].idxmax()
    categoria_mais_similar = similaridades_df['Categoria'][idx_max_similaridade]
    descricao_mais_similar = similaridades_df['Descricao'][idx_max_similaridade]

    if verbose:
        print('------------------')
        print('Tabela de similaridades:')
        print(similaridades_df)
        print('------------------')
        print(f'Índice da descrição mais similar: {idx_max_similaridade}')
        print(f'Categoria mais similar: {categoria_mais_similar}')
        print(f'Descrição mais similar: {descricao_mais_similar}')
        print('------------------')

    return categoria_mais_similar

In [8]:
classificar('peça não funcionou.', verbose=True)

------------------
Tabela de similaridades:
    Similaridade Categoria                                          Descricao
0       0.401793  positivo     Comentários positivos que expressam satisfação
1       0.433089  positivo    Chegou antes do prazo e a qualidade é excelente
2       0.527694  positivo           Excelente acabamento e encaixe perfeito.
3       0.471171  positivo                  Ótima resistência e durabilidade.
4       0.487602  positivo               Produto superou minhas expectativas.
5       0.454600  positivo                Muito eficiente e fácil de instalar
6       0.480842  positivo                             Ótimo custo-benefício.
7       0.438490  positivo                 Entrega rápida e embalagem segura.
8       0.414922  positivo                     atendimento a cima do esperado
9       0.392424  positivo                                        empresa boa
10      0.548217  positivo                                     Peça excelente
11      0.469590  po

'negativo'

In [9]:
classificar('O produto foi otimo', verbose=True)

------------------
Tabela de similaridades:
    Similaridade Categoria                                          Descricao
0       0.640910  positivo     Comentários positivos que expressam satisfação
1       0.730531  positivo    Chegou antes do prazo e a qualidade é excelente
2       0.671718  positivo           Excelente acabamento e encaixe perfeito.
3       0.621066  positivo                  Ótima resistência e durabilidade.
4       0.798377  positivo               Produto superou minhas expectativas.
5       0.616543  positivo                Muito eficiente e fácil de instalar
6       0.682746  positivo                             Ótimo custo-benefício.
7       0.602980  positivo                 Entrega rápida e embalagem segura.
8       0.642294  positivo                     atendimento a cima do esperado
9       0.485806  positivo                                        empresa boa
10      0.606719  positivo                                     Peça excelente
11      0.843015  po

'positivo'

In [10]:
PATH_COMMENTS = '/content/dataset_pecas_industriais.xlsx'

df = pd.read_excel(PATH_COMMENTS)

df.head()

,topico,comentario,sentimento
0,Eixos,Funcionamento silencioso e preciso.,positivo
1,Parafusos,"Compatível com outros equipamentos, recomendo.",positivo
2,Parafusos,Funcionamento silencioso e preciso.,positivo
3,Sensores,Funcionamento silencioso e preciso.,positivo
4,Flanges,Material robusto e confiável.,positivo


In [11]:
df['sentimento'].value_counts()

,count
sentimento,
positivo,260
negativo,240


In [12]:
df.sentimento = df.sentimento.apply(str.lower)

In [13]:
df['sentimento'] = df['sentimento'].replace('postivo', 'positivo')

In [14]:
df['sentimento'].unique()

array(['positivo', 'negativo'], dtype=object)

In [15]:
df = df[df['sentimento'].isin(['positivo', 'negativo'])]

In [16]:
df['sentimento'].value_counts()

,count
sentimento,
positivo,260
negativo,240


In [17]:
y_preds = []
y_tests = []
errors = []
for index, current_row in df.iterrows():
    print('------------------------')
    print(f'Comentário: {current_row["comentario"]}')
    print(f'Sentimento (valor real): {current_row["sentimento"]}')
    y_pred = classificar(current_row['comentario'])
    print(f'Sentimento previsto: {y_pred}')
    y_test = current_row['sentimento']
    y_preds.append(y_pred)
    y_tests.append(y_test)

    if y_pred != y_test:
        missclassification = {}
        missclassification['index'] = index
        missclassification['comentario'] = current_row['comentario']
        missclassification['sentimento_real'] = current_row['sentimento']
        missclassification['sentimento_previsto'] = y_pred
        errors.append(missclassification)

------------------------
Comentário: Funcionamento silencioso e preciso.
Sentimento (valor real): positivo
Sentimento previsto: positivo
------------------------
Comentário: Compatível com outros equipamentos, recomendo.
Sentimento (valor real): positivo
Sentimento previsto: positivo
------------------------
Comentário: Funcionamento silencioso e preciso.
Sentimento (valor real): positivo
Sentimento previsto: positivo
------------------------
Comentário: Funcionamento silencioso e preciso.
Sentimento (valor real): positivo
Sentimento previsto: positivo
------------------------
Comentário: Material robusto e confiável.
Sentimento (valor real): positivo
Sentimento previsto: positivo
------------------------
Comentário: Preço alto para a qualidade entregue.
Sentimento (valor real): negativo
Sentimento previsto: negativo
------------------------
Comentário: Ótima resistência e durabilidade.
Sentimento (valor real): positivo
Sentimento previsto: positivo
------------------------
Comentário:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
# Matriz de Confusão
cm = confusion_matrix(y_tests, y_preds)
print("Matriz de Confusão:")
print(cm)
# Classification Report
cr = classification_report(y_tests, y_preds)
print("\nClassification Report:")
print(cr)
# Acurácia
accuracy = accuracy_score(y_tests, y_preds)
print("\nAcurácia:", accuracy)

Matriz de Confusão:
[[220  20]
 [  0 260]]

Classification Report:
              precision    recall  f1-score   support

    negativo       1.00      0.92      0.96       240
    positivo       0.93      1.00      0.96       260

    accuracy                           0.96       500
   macro avg       0.96      0.96      0.96       500
weighted avg       0.96      0.96      0.96       500


Acurácia: 0.96


In [ ]:
!pip install gradio -q

In [18]:
def classificar_gr(comentario_paciente, verbose=False):
    embedding_paciente = np.array(gerar_embeddings(comentario_paciente)).reshape(1, -1)

    # Calcula a similaridade de cosseno para cada embedding
    similaridades = [cosine_similarity(embedding_paciente, np.array(embedding).reshape(1, -1))[0][0]
                     for embedding in embeddings_sentimentos]

    # Converte a lista de similaridades em um dataframe do pandas para facilitar a análise
    similaridades_df = pd.DataFrame({
        'Similaridade': similaridades,
        'Categoria': categoria_mapping,
        'Descricao': descricao_mapping
    })

    # Encontrando a categoria e descrição com maior similaridade
    idx_max_similaridade = similaridades_df['Similaridade'].idxmax()
    categoria_mais_similar = similaridades_df['Categoria'][idx_max_similaridade]
    descricao_mais_similar = similaridades_df['Descricao'][idx_max_similaridade]

    if verbose:
        print('------------------')
        print('Tabela de similaridades:')
        print(similaridades_df)
        print('------------------')
        print(f'Índice da descrição mais similar: {idx_max_similaridade}')
        print(f'Categoria mais similar: {categoria_mais_similar}')
        print(f'Descrição mais similar: {descricao_mais_similar}')
        print('------------------')

    similaridade_max = similaridades_df['Similaridade'][idx_max_similaridade]
    return categoria_mais_similar, descricao_mais_similar, similaridade_max

In [ ]:
def gradio_classificar_comentarios(comentario_paciente):
    print(comentario_paciente)
    resultado = classificar_gr(comentario_paciente)
    print(resultado[0])
    print(resultado[1])
    print(resultado[2])
    #return f'''O sentimento do comentário do paciente é: {resultado[0]}''', f'''\nO comentário foi categorizado como: {resultado[1]}.''', f'''\nO nível de confiança da predição é: {resultado[2]}'''
    return resultado[0], resultado[1], str(round(resultado[2],2))

demo = gr.Interface(
    fn=gradio_classificar_comentarios,
    inputs=[gr.TextArea(label = 'Entre com o comentário sobre a peça')],
    outputs=[gr.Text(label = 'Sentimento Previsto:'),
             gr.Text(label = 'Categorização de comentário:'),
             gr.Text(label = 'Nível de confiança da predição:')],
    title="Sistema de Análise de Sentimentos de Comentários de peças",
    description="Você deve copiar e colar um comentário de um produto ou digitá-lo na íntegra."
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3464e7b5cc522ecc5f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


as peças chegou a tempo, mais com defeito
negativo
peça não chegou no prazopeça quebrou com pouco tempo de uso
0.79458845
as peças chegou a tempo, mais veio com defeito
negativo
peça não chegou no prazopeça quebrou com pouco tempo de uso
0.777465
fiquei muito feliz com as peças, chegou no prazo
positivo
Chegou antes do prazo e a qualidade é excelente
0.8732388
